In [1]:
from dotenv import load_dotenv
load_dotenv()

False

In [2]:

import os
from llama_index.core import Settings
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.dashscope import DashScopeEmbedding, DashScopeTextEmbeddingModels

# 增加调试日志
import logging
import sys
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
logging.getLogger("llama_index").addHandler(logging.StreamHandler(stream=sys.stdout))


Settings.llm = OpenAILike(
    model="qwen-plus",
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    is_chat_model=True
)

Settings.embed_model = DashScopeEmbedding(
    model_name=DashScopeTextEmbeddingModels.TEXT_EMBEDDING_V3,
    embed_batch_size=6,
    embed_input_length=8192
)

documents = SimpleDirectoryReader("data").load_data()



None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


> [SimpleDirectoryReader] Total files added: 1
DEBUG:llama_index.core.readers.file.base:> [SimpleDirectoryReader] Total files added: 1
DEBUG:fsspec.local:open file: /Users/leimao/Documents/study-code/ai-engineer-training/week03/code/data/公司员工休假福利管理制度.txt


In [3]:
print(documents)

[Document(id_='a2b5ce46-1865-41fb-8535-be94063d59f5', embedding=None, metadata={'file_path': '/Users/leimao/Documents/study-code/ai-engineer-training/week03/code/data/公司员工休假福利管理制度.txt', 'file_name': '公司员工休假福利管理制度.txt', 'file_type': 'text/plain', 'file_size': 6163, 'creation_date': '2026-01-21', 'last_modified_date': '2026-01-21'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='# 公司员工休假福利管理制度\n\n## 第一章 总则\n\n**第一条 目的**  \n为规范公司员工休假管理，保障员工合法权益，维护正常的工作秩序，提升员工满意度与工作效率，根据《中华人民共和国劳动法》《职工带薪年休假条例》《女职工劳动保护特别规定》等国家相关法律法规，结合公司实际情况，制定本制度。\n\n**第二条 适用范围**  \n本制度适用于公司所有与公司签订劳动合同的正式员工（含试用期员工），劳务派遣人员参照执行，具体以劳务派遣协议为准。\n\n**第三条 基本原则**  \n1. 依法合规：严格遵守国家法律法规

In [18]:
# 添加一个新的 Document
from llama_index.core import Document
text = "CEO 可以直接请假，无需向直接领导汇报"

doc = Document(
    text = text,
    metadata = {
        "author": "wilson yin",
        "title": "CEO 请假申请",
        "id": "1234567890"
    }
)

In [19]:
print(doc)

Doc ID: fbdee26a-ca6a-49b8-8672-c58e95601cdf
Text: CEO 可以直接请假，无需向直接领导汇报


In [20]:
#  手动切分Documents

from  llama_index.core.schema import TextNode

n1 = TextNode(text=doc.text[0:10],doc_id=doc.id_)
n2 = TextNode(text=doc.text[10:16],doc_id=doc.id_)

print(n1)
print(n2)

Node ID: 71190413-c0f9-4e15-b6ab-b37465c326df
Text: CEO 可以直接请假
Node ID: 572ca5b0-9c51-40fc-9eac-8ab523db04cf
Text: ，无需向直接


In [ ]:
from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core import Document

doc = Document(
    text=("""
    ### 第七条 事假
    1. 员工因私事必须本人处理的，可申请事假。
    2. 事假需提前申请并获直属主管批准，紧急情况可事后补办手续。
    3. 事假为无薪假，按日扣除相应工资。
    4. 每月事假原则上不超过3天，全年累计不超过15天，特殊情况需经人力资源部及公司领导审批。
    """
    ),
    metadata={"title": "Vacation Questions"}
)

# 内置切分器

splitter = TokenTextSplitter(
    chunk_size=64,
    chunk_overlap=4,
    separator="\n"
)

nodes = splitter.get_nodes_from_documents([doc])

for node in nodes:
    print(node.text)
    print(node.metadata)


Metadata length (6) is close to chunk size (32). Resulting chunks are less than 50 tokens. Consider increasing the chunk size or decreasing the size of your metadata to avoid this.
### 第七条 事假
    1. 员工因私事必
{'title': 'Vacation Questions'}
因私事必须本人处理的，可申请事假。
    2
{'title': 'Vacation Questions'}
2. 事假需提前申请并获直属主管批
{'title': 'Vacation Questions'}
主管批准，紧急情况可事后补办手续。
{'title': 'Vacation Questions'}
续。
    3. 事假为无薪假，按日扣
{'title': 'Vacation Questions'}
按日扣除相应工资。
    4. 每月事假原则上
{'title': 'Vacation Questions'}
原则上不超过3天，全年累计不超过15天，特殊情
{'title': 'Vacation Questions'}
特殊情况需经人力资源部及公司领导审批。
{'title': 'Vacation Questions'}


In [9]:
# 测试：当内容超过 chunk_size 时，chunk_overlap 是否生效

from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core import Document

# 创建一个长文本，确保某些段落会超过 chunk_size
doc = Document(
    text=("""
    这是第一段。这是一个很短的段落。

    这是第二段。这是一个非常长的段落，需要包含足够多的内容来超过 chunk_size=64 tokens 的限制。我们需要添加更多的文字来确保这个段落会被切分。让我们继续添加内容，直到这个段落足够长，能够触发切分机制。这样我们就能看到 chunk_overlap 是否真的生效了。我们需要更多的文字来达到这个目标。

    这是第三段。这是另一个段落。
    """
    ),
    metadata={"title": "Overlap Test"}
)

# 使用较小的 chunk_size 来确保会触发切分
splitter = TokenTextSplitter(
    chunk_size=64,
    chunk_overlap=8,  # 增加 overlap 以便更容易观察
    separator="\n"
)

nodes = splitter.get_nodes_from_documents([doc])

print(f"总共切分为 {len(nodes)} 个块：\n")
for i, node in enumerate(nodes, 1):
    print(f"【块 {i}】")
    print(f"内容: {node.text[:100]}...")  # 只显示前100个字符
    print(f"长度: {len(node.text)} 字符")
    print("-" * 50)

# 直接检查原始文本，看切分点
full_text = doc.text
print("原始文本长度:", len(full_text), "字符")

# 检查每个块在原始文本中的位置
for i, node in enumerate(nodes):
    start_pos = full_text.find(node.text[:20])  # 找到块在原文中的位置
    if start_pos != -1:
        print(f"\n块 {i+1}:")
        print(f"  在原文中的位置: {start_pos}")
        print(f"  块长度: {len(node.text)} 字符")
        if i > 0:
            prev_end = full_text.find(nodes[i-1].text[-20:]) + len(nodes[i-1].text[-20:])
            overlap = start_pos - prev_end
            print(f"  与前一块的间隔: {overlap} 字符")
            if overlap < 0:
                print(f"  ✓ 有重叠！重叠了 {-overlap} 个字符")

总共切分为 5 个块：

【块 1】
内容: 这是第一段。这是一个很短的段落。

    这是第二段。这是一个非常长的段落，需要包含足够多的内...
长度: 48 字符
--------------------------------------------------
【块 2】
内容: 含足够多的内容来超过 chunk_size=64 tokens 的限制。我们需要添加更多的文字来确保这个段...
长度: 53 字符
--------------------------------------------------
【块 3】
内容: 文字来确保这个段落会被切分。让我们继续添加内容，直到这个段落足够长，能够触发切分机制。这...
长度: 44 字符
--------------------------------------------------
【块 4】
内容: 发切分机制。这样我们就能看到 chunk_overlap 是否真的生效了。我们需要更多的文字来达到这个目标。...
长度: 54 字符
--------------------------------------------------
【块 5】
内容: 达到这个目标。

    这是第三段。这是另一个段落。...
长度: 27 字符
--------------------------------------------------
原始文本长度: 208 字符

块 1:
  在原文中的位置: 5
  块长度: 48 字符

块 2:
  在原文中的位置: 47
  块长度: 53 字符
  与前一块的间隔: -6 字符
  ✓ 有重叠！重叠了 6 个字符

块 3:
  在原文中的位置: 92
  块长度: 44 字符
  与前一块的间隔: -8 字符
  ✓ 有重叠！重叠了 8 个字符

块 4:
  在原文中的位置: 129
  块长度: 54 字符
  与前一块的间隔: -7 字符
  ✓ 有重叠！重叠了 7 个字符

块 5:
  在原文中的位置: 176
  块长度: 27 字符
  与前一块的间隔: -7 字符
  ✓ 有重叠！重叠了 7 个字符


In [8]:
# 对比：没有 separator 时，chunk_overlap 的行为

from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core import Document

# 创建一个连续的长文本（没有换行符）
doc = Document(
    text="这是一个非常长的连续文本，没有任何分隔符。我们需要确保这个文本足够长，超过 chunk_size=64 tokens 的限制。这样我们就能清楚地看到 chunk_overlap 是如何工作的。让我们继续添加更多的文字，直到这个文本足够长，能够被切分成多个块。我们需要更多的内容来触发切分机制，这样就能观察到重叠效果了。继续添加文字，直到达到目标长度。",
    metadata={"title": "No Separator Test"}
)

# 不使用 separator，直接按 token 切分
# 注意：TokenTextSplitter 不允许空字符串作为 separator
# 如果不设置 separator 参数，它会使用默认的分隔符（通常是空格）
# 由于我们的测试文本是连续的，没有空格，所以会按 token 数量直接切分
splitter_no_sep = TokenTextSplitter(
    chunk_size=64,
    chunk_overlap=8
    # 不设置 separator 参数，使用默认行为
)

nodes_no_sep = splitter_no_sep.get_nodes_from_documents([doc])

print("【没有 separator 的情况】")
print(f"总共切分为 {len(nodes_no_sep)} 个块：\n")
for i, node in enumerate(nodes_no_sep, 1):
    print(f"【块 {i}】")
    print(f"内容: {node.text}")
    print(f"长度: {len(node.text)} 字符")
    print("-" * 50)
# 直接检查原始文本，看切分点
full_text = doc.text
print("原始文本长度:", len(full_text), "字符")

# 检查每个块在原始文本中的位置
for i, node in enumerate(nodes_no_sep):
    start_pos = full_text.find(node.text[:20])  # 找到块在原文中的位置
    if start_pos != -1:
        print(f"\n块 {i+1}:")
        print(f"  在原文中的位置: {start_pos}")
        print(f"  块长度: {len(node.text)} 字符")
        if i > 0:
            prev_end = full_text.find(nodes_no_sep[i-1].text[-20:]) + len(nodes_no_sep[i-1].text[-20:])
            overlap = start_pos - prev_end
            print(f"  与前一块的间隔: {overlap} 字符")
            if overlap < 0:
                print(f"  ✓ 有重叠！重叠了 {-overlap} 个字符")

【没有 separator 的情况】
总共切分为 4 个块：

【块 1】
内容: 这是一个非常长的连续文本，没有任何分隔符。我们需要确保这个文本足够长，超过 chunk_size=64 tokens 的限制。这样我们就能清楚地看到
长度: 74 字符
--------------------------------------------------
【块 2】
内容: chunk_overlap 是如何工作的。让我们继续添加更多的文字，直到这个文本足够长，能够被切分成多个块。我们需要
长度: 58 字符
--------------------------------------------------
【块 3】
内容: 个块。我们需要更多的内容来触发切分机制，这样就能观察到重叠效果了。继续添加文字，直到达到目标长度
长度: 48 字符
--------------------------------------------------
【块 4】
内容: 直到达到目标长度。
长度: 9 字符
--------------------------------------------------
原始文本长度: 175 字符

块 1:
  在原文中的位置: 0
  块长度: 74 字符

块 2:
  在原文中的位置: 75
  块长度: 58 字符
  与前一块的间隔: 1 字符

块 3:
  在原文中的位置: 126
  块长度: 48 字符
  与前一块的间隔: -7 字符
  ✓ 有重叠！重叠了 7 个字符

块 4:
  在原文中的位置: 166
  块长度: 9 字符
  与前一块的间隔: -8 字符
  ✓ 有重叠！重叠了 8 个字符
